In [ ]:
!pip install transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=915fdd4f3c9e07606e28695f9d514da6f1e752d044beffe99682bb842d2e5e9a
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# IMPORTS
import json
import os
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

In [ ]:
# CONFIG
MODEL_NAME   = "classla/bcms-bertic"
DATASET_PATH = "train.txt"
OUTPUT_DIR   = "./bertic-olx"
SEED         = 42

LABELS = [
    "O",
    "B-BRAND", "I-BRAND",
    "B-MOD", "I-MOD",
    "B-MEM", "I-MEM",
    "B-SIM", "I-SIM",
    "B-BATT", "I-BATT",
    "B-BOX", "I-BOX",
    "B-COND", "I-COND",
    "B-FAIL", "I-FAIL",
    "B-ICL", "I-ICL"
]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

In [ ]:
def load_conll(path: str):
    sentences = []
    tokens, tags = [], []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": tags})
                    tokens, tags = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                tags.append(parts[1])
        if tokens:
            sentences.append({"tokens": tokens, "ner_tags": tags})

    return sentences

def tokenize_and_align(examples, tokenizer):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True,   #
    )

    aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

def make_compute_metrics(label_names):
    seqeval = evaluate.load("seqeval")

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = np.argmax(logits, axis=-1)

        true_labels = [
            [label_names[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [label_names[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"],
        }

    return compute_metrics

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# load and split
raw_data = load_conll(DATASET_PATH)

np.random.seed(SEED)
np.random.shuffle(raw_data)
split = int(len(raw_data) * 0.8)

train_records = raw_data[:split]
val_records   = raw_data[split:]

print(f"Train: {len(train_records)}, Val: {len(val_records)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Train: 124, Val: 31


In [ ]:
def diagnose(records, name):
    total = 0
    entity = 0
    label_counts = {}
    for r in records:
        for t in r["ner_tags"]:
            total += 1
            if t != "O":
                entity += 1
            label_counts[t] = label_counts.get(t, 0) + 1

    print(f"\n── {name} ──")
    print(f"Total tokens: {total}")
    print(f"Entity tokens: {entity} ({100*entity/total:.1f}%)")
    print(f"O tokens: {total-entity} ({100*(total-entity)/total:.1f}%)")
    print("Per label:", dict(sorted(label_counts.items())))

diagnose(train_records, "TRAIN")
diagnose(val_records, "VALIDATION")


── TRAIN ──
Total tokens: 2544
Entity tokens: 1306 (51.3%)
O tokens: 1238 (48.7%)
Per label: {'B-BATT': 104, 'B-BOX': 52, 'B-BRAND': 89, 'B-COND': 112, 'B-FAIL': 24, 'B-ICL': 23, 'B-MEM': 120, 'B-MOD': 119, 'B-SIM': 34, 'I-BATT': 105, 'I-BOX': 19, 'I-COND': 134, 'I-FAIL': 31, 'I-ICL': 23, 'I-MEM': 120, 'I-MOD': 170, 'I-SIM': 27, 'O': 1238}

── VALIDATION ──
Total tokens: 592
Entity tokens: 347 (58.6%)
O tokens: 245 (41.4%)
Per label: {'B-BATT': 30, 'B-BOX': 13, 'B-BRAND': 25, 'B-COND': 33, 'B-FAIL': 4, 'B-ICL': 10, 'B-MEM': 31, 'B-MOD': 30, 'B-SIM': 8, 'I-BATT': 30, 'I-BOX': 4, 'I-COND': 38, 'I-FAIL': 4, 'I-ICL': 10, 'I-MEM': 31, 'I-MOD': 38, 'I-SIM': 8, 'O': 245}


In [ ]:
def to_hf(records):
    return Dataset.from_list([
        {
            "tokens":   r["tokens"],
            "ner_tags": [LABEL2ID.get(t, 0) for t in r["ner_tags"]],
        }
        for r in records
    ])

dataset = DatasetDict({
    "train":      to_hf(train_records),
    "validation": to_hf(val_records),
})

tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align(x, tokenizer),
    batched=True,
    remove_columns=["tokens", "ner_tags"],
)
MODEL_NAME = "./bertic"
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=25,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    seed=SEED,
    fp16=False,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(LABELS),
)

print("Starting training...")
trainer.train()

trainer.save_model(f"{OUTPUT_DIR}/best")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
print(f"Saved to {OUTPUT_DIR}/best")



Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./bertic-olx/best


In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="./bertic-olx/checkpoint-400",
    aggregation_strategy="simple"
)

tests = [
    "iPhone 13 Pro 256GB baterija 89% kao nov kutija",
    "Samsung S23 128GB 91% odlično stanje icloud slobodan",
    "iPhone 12 64GB 76% ne radi Face ID mjenjan ekran",
    "Huawei P30 Pro 128GB dual SIM 78% dobro stanje bez kutije",
    "IPHONE 16 PRO 128GB 90% BATERIJA NATURAL TITANIUM",
    "iphone 15 512gb esim baterija 97% kao novo puna kutija",
    "Samsung A54 128GB 86% very good condition box included",
    "iPhone 11 128GB 79% puknut ekran icloud slobodan",
]

print(f"{'─'*60}")
for text in tests:
    results = ner(text)
    print(f"\nINPUT: {text}")
    if results:
        for r in results:
            print(f"  {r['entity_group']:<12} {r['word']:<20} score={r['score']:.2f}")
    else:
        print("  (no entities found)")
    print(f"{'─'*60}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

────────────────────────────────────────────────────────────

INPUT: iPhone 13 Pro 256GB baterija 89% kao nov kutija
  MOD          iPhone 13 Pro        score=0.97
  MEM          25                   score=0.96
  MEM          ##6GB                score=0.92
  BATT         89 %                 score=0.96
  COND         kao nov              score=0.94
  BOX          kutija               score=0.87
────────────────────────────────────────────────────────────

INPUT: Samsung S23 128GB 91% odlično stanje icloud slobodan
  BRAND        Samsung              score=0.96
  MOD          S23                  score=0.97
  MEM          128GB                score=0.96
  BATT         91 %                 score=0.96
  COND         odlično stanje       score=0.94
  ICL          i                    score=0.82
  MOD          ##cloud              score=0.14
  ICL          slobodan             score=0.83
────────────────────────────────────────────────────────────

INPUT: iPhone 12 64GB 76% ne radi Face ID